In [12]:

import subprocess
import os
import concurrent.futures

import threading
import sys

def _filter_stderr(pipe):
    noisy_patterns = [
        "MathWorksServiceHost",
        "GLIBCXX_3.4.31",
        "GLIBCXX_3.4.32",
        "CXXABI_1.3.15",
        "libmwflnetwork.so",
        "libmwflcrypto.so",
        "libmwflcryptoutils.so",
        "libmwflcryptoopenssl.so",
        "libmwflurlmgrfactory.so",
    ]

    for line in pipe:
        if not any(p in line for p in noisy_patterns):
            sys.stderr.write(line)
            sys.stderr.flush()

def run(cmd):
    process = subprocess.Popen(
        cmd,
        shell=True,
        text=True,
        stdout=None,
        stderr=subprocess.PIPE
    )

    t = threading.Thread(target=_filter_stderr, args=(process.stderr,))
    t.daemon = True
    t.start()

    returncode = process.wait()
    t.join(timeout=1)

    if returncode != 0:
        print(f"Command exited with code {returncode}", file=sys.stderr)

def lastFrame(directory):
    frames = [int(file.split("geo")[1].split(".")[0]) for file in os.listdir(directory) if file.startswith("geo") and file.endswith(".mat")]
    return max(frames)

here = "/home/nickbroussinos/Code/evolving_stokes_codex/remesh/"

# Inputs
verbose = 'true'
resume = False
supress_outputs = 0
subdivisions = 4
roughness = .0005
remesh_size = 1
T = 40000
k = 10000
Da = 1000
Sd = 21
Gamma = 0
gamy = 5
chi = 2e-5

param = [.05] # dt values

### ALWAYS CHECK RESUME VALUE ###

commands = []
for dt in param:
        run_tag = f"Sd_{Sd:.2e}_Da_{Da:.2e}_gamy_{gamy:+.2e}".replace('+', 'p').replace('-', 'm')
        dir_path = here + f"data/gs_batch_data/{run_tag}"
        os.makedirs(dir_path, exist_ok=True)
        start = (lastFrame(dir_path)) if resume else 0
        if resume:
            print(f"start: {start}")
        cmd = f"""export LD_PRELOAD=/usr/lib/x86_64-linux-gnu/libstdc++.so.6 && \
    matlab -nodisplay -nosplash -nodesktop -r "cd '{here}'; verbose = {verbose};\
    supress_outputs = {supress_outputs}; dir = '{dir_path}/'; start = {start};\
    p = struct(); p.roughness = {roughness}; p.dt={dt}; p.T={T}; p.start={start};\
    p.subdivisions={subdivisions}; p.k={k}; p.remesh_size = {remesh_size};\
    p.Da={Da}; p.Sd={Sd}; p.Gamma={Gamma}; p.gamy={gamy}; p.chi={chi}; gs_multi; exit" """
        commands.append(cmd)


max_workers = 6
with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(run, cmd) for cmd in commands]
    concurrent.futures.wait(futures)

    


                            < M A T L A B (R) >
                  Copyright 1984-2024 The MathWorks, Inc.
             R2024a Update 3 (24.1.0.2603908) 64-bit (glnxa64)
                                May 2, 2024

 
To get started, type doc.
For product information, visit www.mathworks.com.
 
Save geo0.mat 
    NK scales: residual u = 3.381e+04, f = 1.610e+00, area = 1.233e+01; Schur state P = 5.774e-01, lambda = 1.388e+00
t = 1, j = 0, Schur NK phi = 0.6697, eps_b = 0.0006881, eps_c = 0.9476, eps_d = 3.432e-08, GMRES flag = 0, relres = 0.2715, tol = 0.5, iter = [1 2], alpha = 1
t = 1, j = 1, Schur NK phi = 0.6333, eps_b = 0.00132, eps_c = 0.8961, eps_d = 6.776e-08, GMRES flag = 0, relres = 0.2891, tol = 0.5, iter = [1 2], alpha = 1
t = 1, j = 2, Schur NK phi = 0.5953, eps_b = 0.001932, eps_c = 0.8423, eps_d = 1.013e-07, GMRES flag = 0, relres = 0.305, tol = 0.5, iter = [1 2], alpha = 1
t = 1, j = 3, Schur NK phi = 0.5535, eps_b = 0.002547, eps_c = 0.7832, eps_d = 1.355e-07, GMRES fla